# E-HGP vs HGP-old vs HDBSCAN: Google Colab Benchmark

Ce notebook permet de comparer les performances en temps de calcul et en precision (ARI) de trois algorithmes de clustering :
1. **HGP-old** (Version exacte combinatoire développée dans `HGP-old` utilisant Delaunay/Voronoi global).
2. **E-HGP** (Version regularisee locale et parallélisable développée dans `E-HGP` avec support Cython/CPU).
3. **HDBSCAN** (Version standard basée sur la densité).

## Configuration Google Colab :
Afin de profiter de l'accélération matérielle, allez dans **Runtime** -> **Change runtime type** -> Sélectionnez **T4 GPU** (ou GPU supérieur si Colab Pro).

### 1. Installation des dépendances et compilation des packages

In [ ]:
# Installation des dependances systeme (Eigen3 et CGAL pour HGP-old)
!apt-get update && apt-get install -y libeigen3-dev libcgal-dev

# Installation des packages python requis
!pip install pybind11 Cython hdbscan gudhi

# Recuperation du depot GitHub de la these
!git clone https://github.com/Ludwig-H/E-HGP.git

# Compilation de HGP-old (version combinatoire) avec chemin absolu et sans isolation
!pip install --no-build-isolation /content/E-HGP/HGP-old

# Compilation de E-HGP (version entropique optimisee) avec chemin absolu
!pip install /content/E-HGP/E-HGP

### 2. Imports et vérification du matériel

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_blobs
from sklearn.metrics import adjusted_rand_score

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositif de calcul detecte : {device.upper()}')

import e_hgp
import hgp_clusterer
from hdbscan import HDBSCAN

print('e_hgp importe avec succes !')
print('hgp_clusterer importe avec succes !')

### 3. Implémentation du framework de Benchmark
Nous allons concevoir des scénarios variés :
* **Variabilité en dimension** : 2D, 3D, 10D, 30D.
* **Variabilité en nombre de points** : 500 à 50 000 points.
* **Variabilité des hyperparamètres** : Ordre $K$, régularisation et taille minimale des clusters.

In [ ]:
def run_single_experiment(X, y, algo_name, params):
    t0 = time.time()
    try:
        if algo_name == 'HGP-old':
            if X.shape[1] > 3:
                return None, None
            model = hgp_clusterer.HGPClusterer(
                K=params.get('K', 2), 
                min_cluster_size=params.get('min_cluster_size', 15),
                expZ=params.get('expZ', 2.0)
            )
            labels = model.fit_predict(X)
        elif algo_name == 'E-HGP':
            model = e_hgp.EHGPClusterer(
                K=params.get('K', 2),
                kappa=params.get('kappa', 1.5),
                m_reg=params.get('m_reg', 25),
                min_cluster_size=params.get('min_cluster_size', 15),
                expZ=params.get('expZ', 2.0)
            )
            labels = model.fit_predict(X)
        elif algo_name == 'HDBSCAN':
            model = HDBSCAN(min_cluster_size=params.get('min_cluster_size', 15))
            labels = model.fit_predict(X)
        else:
            raise ValueError('Algo inconnu')
        
        t_exec = time.time() - t0
        ari = adjusted_rand_score(y, labels)
        return t_exec, ari
    except Exception as e:
        print(f'Erreur lors de l\'execution de {algo_name}: {e}')
        return None, None

### 4. Scénario 1 : Comparaison en 2D (Nesting Moons) et 3D (Blobs) avec variation de taille ($N$)
Ce test compare E-HGP à HGP-old et HDBSCAN.

In [ ]:
sizes = [500, 1000, 2500, 5000, 10000]
algos = ['E-HGP', 'HGP-old', 'HDBSCAN']

results_2d = {algo: {'times': [], 'aris': []} for algo in algos}
results_3d = {algo: {'times': [], 'aris': []} for algo in algos}

for N in sizes:
    print(f'Benchmark N = {N}...')
    X_2d, y_2d = make_moons(n_samples=N, noise=0.08, random_state=42)
    X_3d, y_3d = make_blobs(n_samples=N, centers=3, n_features=3, random_state=42)
    
    min_cluster_size = max(10, N // 100)
    params = {'K': 2, 'kappa': 1.5, 'min_cluster_size': min_cluster_size, 'expZ': 2.0}
    
    for algo in algos:
        # Run 2D
        t, ari = run_single_experiment(X_2d, y_2d, algo, params)
        if t is not None:
            results_2d[algo]['times'].append(t)
            results_2d[algo]['aris'].append(ari)
        else:
            results_2d[algo]['times'].append(np.nan)
            results_2d[algo]['aris'].append(0.0)
            
        # Run 3D
        t, ari = run_single_experiment(X_3d, y_3d, algo, params)
        if t is not None:
            results_3d[algo]['times'].append(t)
            results_3d[algo]['aris'].append(ari)
        else:
            results_3d[algo]['times'].append(np.nan)
            results_3d[algo]['aris'].append(0.0)

print('Scenario 1 termine !')

#### Visualisation graphique des temps de calcul et de la précision (2D & 3D)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 2D Times
for algo in algos:
    axes[0, 0].plot(sizes, results_2d[algo]['times'], marker='o', label=algo)
axes[0, 0].set_title('Temps de calcul vs Taille (2D Moons)')
axes[0, 0].set_xlabel('N points')
axes[0, 0].set_ylabel('Temps (s)')
axes[0, 0].set_yscale('log')
axes[0, 0].grid(True)
axes[0, 0].legend()

# 2D ARI
for algo in algos:
    axes[0, 1].plot(sizes, results_2d[algo]['aris'], marker='s', label=algo)
axes[0, 1].set_title('Precision (ARI) vs Taille (2D Moons)')
axes[0, 1].set_xlabel('N points')
axes[0, 1].set_ylabel('ARI')
axes[0, 1].set_ylim(0.8, 1.05)
axes[0, 1].grid(True)
axes[0, 1].legend()

# 3D Times
for algo in algos:
    axes[1, 0].plot(sizes, results_3d[algo]['times'], marker='o', label=algo)
axes[1, 0].set_title('Temps de calcul vs Taille (3D Blobs)')
axes[1, 0].set_xlabel('N points')
axes[1, 0].set_ylabel('Temps (s)')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True)
axes[1, 0].legend()

# 3D ARI
for algo in algos:
    axes[1, 1].plot(sizes, results_3d[algo]['aris'], marker='s', label=algo)
axes[1, 1].set_title('Precision (ARI) vs Taille (3D Blobs)')
axes[1, 1].set_xlabel('N points')
axes[1, 1].set_ylabel('ARI')
axes[1, 1].set_ylim(0.8, 1.05)
axes[1, 1].grid(True)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### 5. Scénario 2 : Haute dimension (nD) - E-HGP vs HDBSCAN
Ici, HGP-old n'est pas testé car la construction de la mosaïque Delaunay échoue en dimension $>3$.

In [ ]:
dims = [2, 5, 10, 20, 50]
N_nd = 2000
algos_nd = ['E-HGP', 'HDBSCAN']

results_nd = {algo: {'times': [], 'aris': []} for algo in algos_nd}

for d in dims:
    print(f'Benchmark dimension d = {d}...')
    X_nd, y_nd = make_blobs(n_samples=N_nd, centers=4, n_features=d, random_state=42)
    params = {'K': 2, 'kappa': 1.5, 'min_cluster_size': 20, 'expZ': 2.0}
    
    for algo in algos_nd:
        t, ari = run_single_experiment(X_nd, y_nd, algo, params)
        results_nd[algo]['times'].append(t if t is not None else np.nan)
        results_nd[algo]['aris'].append(ari if ari is not None else 0.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for algo in algos_nd:
    axes[0].plot(dims, results_nd[algo]['times'], marker='o', label=algo)
axes[0].set_title(f'Temps de calcul vs Dimension (N={N_nd})')
axes[0].set_xlabel('Dimension d')
axes[0].set_ylabel('Temps (s)')
axes[0].set_yscale('log')
axes[0].grid(True)
axes[0].legend()

for algo in algos_nd:
    axes[1].plot(dims, results_nd[algo]['aris'], marker='s', label=algo)
axes[1].set_title(f'Precision (ARI) vs Dimension (N={N_nd})')
axes[1].set_xlabel('Dimension d')
axes[1].set_ylabel('ARI')
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(True)
axes[1].legend()

plt.show()

### 6. Scénario 3 : Influence des paramètres $K$ (ordre) et $expZ$ (exposant de distance)
Nous étudions l'impact du réglage de la distance de puissance $expZ$ ($1.0$ comme HDBSCAN, $2.0$ par défaut) et de l'ordre géométrique $K$ sur E-HGP.

In [ ]:
expZ_values = [1.0, 2.0, 3.0]
k_values = [2, 3]
X_param, y_param = make_blobs(n_samples=5000, centers=5, n_features=3, cluster_std=2.0, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for K in k_values:
    aris_z = []
    times_z = []
    for expZ in expZ_values:
        params = {'K': K, 'kappa': 1.5, 'min_cluster_size': 50, 'expZ': expZ}
        t, ari = run_single_experiment(X_param, y_param, 'E-HGP', params)
        aris_z.append(ari if ari is not None else 0.0)
        times_z.append(t if t is not None else 0.0)
    
    axes[0].plot(expZ_values, aris_z, marker='o', label=f'K={K} (ARI)')
    axes[1].plot(expZ_values, times_z, marker='s', label=f'K={K} (Temps)')

axes[0].set_title('Sensibilite au parametre expZ (ARI)')
axes[0].set_xlabel('expZ')
axes[0].set_ylabel('ARI')
axes[0].grid(True)
axes[0].legend()

axes[1].set_title('Temps de calcul vs expZ')
axes[1].set_xlabel('expZ')
axes[1].set_ylabel('Temps (s)')
axes[1].grid(True)
axes[1].legend()

plt.show()

### 7. Scénario 4 : Comparatif GPU vs CPU (Accélération CUDA sur E-HGP)
Ce test compare directement la vitesse de notre module de régularisation entropique et de Gabriel lorsqu'il tourne sur carte graphique (CUDA) vs sur CPU.

In [ ]:
import torch
if not torch.cuda.is_available():
    print('GPU CUDA non disponible, le benchmark GPU vs CPU ne peut pas etre execute de maniere significative.')
else:
    sizes_gpu = [1000, 5000, 10000, 25000, 50000]
    times_cpu = []
    times_gpu = []
    
    original_is_available = torch.cuda.is_available
    
    for N in sizes_gpu:
        print(f'Test GPU vs CPU pour N = {N}...')
        X_gpu, y_gpu = make_blobs(n_samples=N, centers=5, n_features=10, random_state=42)
        params = {'K': 2, 'kappa': 1.5, 'min_cluster_size': N // 100}
        
        # 1. Force CPU mode
        torch.cuda.is_available = lambda: False
        t0 = time.time()
        model_cpu = e_hgp.EHGPClusterer(
            K=params['K'],
            kappa=params['kappa'],
            min_cluster_size=params['min_cluster_size']
        )
        model_cpu.fit(X_gpu)
        times_cpu.append(time.time() - t0)
        
        # 2. Force GPU mode
        torch.cuda.is_available = original_is_available
        t0 = time.time()
        model_gpu = e_hgp.EHGPClusterer(
            K=params['K'],
            kappa=params['kappa'],
            min_cluster_size=params['min_cluster_size']
        )
        model_gpu.fit(X_gpu)
        times_gpu.append(time.time() - t0)
        
    # Plot the results
    plt.figure(figsize=(10, 6))
    plt.plot(sizes_gpu, times_cpu, marker='o', label='CPU Fallback')
    plt.plot(sizes_gpu, times_gpu, marker='s', label='GPU (CUDA Accelerated)')
    plt.title('Temps de calcul E-HGP : GPU vs CPU')
    plt.xlabel('Nombre de points N')
    plt.ylabel('Temps (s)')
    plt.yscale('log')
    plt.grid(True)
    plt.legend()
    plt.show()

### 8. Scénario 5 : Passage à l'échelle hypermassif (10 000 000 de points)
Ce test compare E-HGP et HDBSCAN sur un dataset massif de 10 millions de points en 3D avec 100 clusters. Il met en évidence la robustesse mémoire de E-HGP (avec chunking dynamique et L_max=30) face à HDBSCAN qui sature la mémoire (OOM).

In [ ]:
print('Generation d\'un dataset hypermassif : 10 000 000 de points en 3D avec 100 clusters...')
t0 = time.time()
X_mass, y_mass = make_blobs(n_samples=10000000, centers=100, n_features=3, random_state=42)
print(f'Dataset genere en {time.time() - t0:.2f}s.')

import torch
original_is_available = torch.cuda.is_available

print('=== RUNNING E-HGP on CPU (10M points) ===')
torch.cuda.is_available = lambda: False
t0 = time.time()
ehgp_cpu = e_hgp.EHGPClusterer(K=2, kappa=1.5, m_reg=20, min_cluster_size=1000, L_initial=10)
labels_cpu = ehgp_cpu.fit_predict(X_mass)
t_cpu = time.time() - t0
sub_idx = np.random.choice(10000000, size=50000, replace=False)
ari_cpu = adjusted_rand_score(y_mass[sub_idx], labels_cpu[sub_idx])
print(f'E-HGP (CPU) termine en {t_cpu:.2f}s | ARI (subsample 50k): {ari_cpu:.4f}')

print('=== RUNNING E-HGP on GPU (10M points) ===')
torch.cuda.is_available = original_is_available
if not torch.cuda.is_available():
    print('GPU CUDA non disponible, execution GPU ignoree.')
    t_gpu = float('nan')
    ari_gpu = 0.0
else:
    t0 = time.time()
    ehgp_gpu = e_hgp.EHGPClusterer(K=2, kappa=1.5, m_reg=20, min_cluster_size=1000, L_initial=10)
    labels_gpu = ehgp_gpu.fit_predict(X_mass)
    t_gpu = time.time() - t0
    ari_gpu = adjusted_rand_score(y_mass[sub_idx], labels_gpu[sub_idx])
    print(f'E-HGP (GPU) termine en {t_gpu:.2f}s | ARI (subsample 50k): {ari_gpu:.4f}')

print('=== RUNNING HDBSCAN (10M points) ===')
try:
    t0 = time.time()
    hdb_mass = HDBSCAN(min_cluster_size=1000, core_dist_n_jobs=-1)
    labels_hdb_mass = hdb_mass.fit_predict(X_mass)
    t_hdb_mass = time.time() - t0
    ari_hdb_mass = adjusted_rand_score(y_mass[sub_idx], labels_hdb_mass[sub_idx])
    print(f'HDBSCAN termine en {t_hdb_mass:.2f}s | ARI (subsample 50k): {ari_hdb_mass:.4f}')
except Exception as e:
    print(f'HDBSCAN a echoue ou a manque de memoire (OOM) : {e}')
    t_hdb_mass = float('nan')

print("\n" + "="*80)
print("RAPPORT COMPARATIF SUR NUAGES SUPER-MASSIFS (10 000 000 points)")
print("="*80)
print(f"E-HGP (CPU) : {t_cpu:.2f}s | ARI = {ari_cpu:.4f}")
if not np.isnan(t_gpu):
    print(f"E-HGP (GPU) : {t_gpu:.2f}s | ARI = {ari_gpu:.4f} (Speedup: {t_cpu/t_gpu:.1f}x)")
else:
    print("E-HGP (GPU) : NON EXECUTE (Pas de GPU)")
if not np.isnan(t_hdb_mass):
    print(f"HDBSCAN     : {t_hdb_mass:.2f}s")
else:
    print("HDBSCAN     : ECHEC (OOM)")
print("="*80 + "\n")
